# VIIRS Flood Timelapse — NOAA AWS Only (Personal Project Version)

Builds a timelapse (GIF/MP4) from NOAA JPSS VIIRS-ABI Flood Day granules, pulled directly from the public
`noaa-jpss` S3 bucket (anonymous access, no credentials, no STAC needed).

This is the pre-gap half of a larger project that eventually bridges to a second, post-gap archive via a STAC
catalog — that part is deliberately **left out for now**. This notebook proves the pipeline on one source first:
fetch → decode → normalize → render with date labels → animate.


In [25]:
# --- Setup -----------------------------------------------------------------
import os, urllib.request, urllib.parse, xml.etree.ElementTree as ET

import numpy as np
import xarray as xr
!pip install rioxarray
import rioxarray as rxr

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from PIL import Image, ImageDraw, ImageFont

DATA_DIR = "data"
FRAMES_DIR = "frames"
OUT_DIR = "output"
for d in (DATA_DIR, FRAMES_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

# AOI (Amazon basin bbox, WGS84 lon/lat) — swap for your own basin if needed.
AOI_BBOX = (-58.0, -20.0, -53.0, -15.0)  # (minx, miny, maxx, maxy) — inside the Pantanal, within this granule's coverage
COMMON_CRS = "EPSG:4326"


## Manifest

Pin the exact granules used, so re-running this notebook regenerates the same animation. Start with just one
date, confirm it renders correctly, then add more.


In [26]:
NOAA_PREFIX = "JPSS_Blended_Products/VIIRS-ABI-Flood-Day/{yyyy}/{mm}/{dd}/"

NOAA_MANIFEST = [
    # (date, filename) — known-good Amazon-basin granule with a clear flood signal
    ("2025-09-15", "VIIRS-ABI-Flood-GLB035_v1r1_blend_s202509151541059_e202509151836032_c202509160739408.nc"),
    # add more dates here once the first one renders correctly
]


## Section 1 — Fetch NOAA granules from public S3

No STAC, no credentials — just a plain HTTPS download from the public Open Data bucket.


In [27]:
def list_noaa_day(yyyy, mm, dd, max_keys=50):
    """List .nc granules for one day in the public noaa-jpss bucket (anonymous, no credentials)."""
    base = "https://noaa-jpss.s3.amazonaws.com/"
    prefix = NOAA_PREFIX.format(yyyy=yyyy, mm=mm, dd=dd)
    query = urllib.parse.urlencode({"list-type": "2", "prefix": prefix, "max-keys": str(max_keys)})
    xml_bytes = urllib.request.urlopen(base + "?" + query).read()
    ns = "{http://s3.amazonaws.com/doc/2006-03-01/}"
    keys = [e.text for e in ET.fromstring(xml_bytes).iter(ns + "Key") if e.text.endswith(".nc")]
    return keys


def download_noaa_granule(key, dest_dir=DATA_DIR):
    base = "https://noaa-jpss.s3.amazonaws.com/"
    url = base + urllib.parse.quote(key)
    local_path = os.path.join(dest_dir, os.path.basename(key))
    if not os.path.exists(local_path):
        urllib.request.urlretrieve(url, local_path)
    return local_path


noaa_local_paths = []
for date_str, filename in NOAA_MANIFEST:
    yyyy, mm, dd = date_str.split("-")
    key = NOAA_PREFIX.format(yyyy=yyyy, mm=mm, dd=dd) + filename
    path = download_noaa_granule(key)
    noaa_local_paths.append((date_str, path))
    print("downloaded:", path)


downloaded: data/VIIRS-ABI-Flood-GLB035_v1r1_blend_s202509151541059_e202509151836032_c202509160739408.nc


In [28]:
ds = xr.open_dataset(noaa_local_paths[0][1])
print(ds)
print("---")
print("coords:", list(ds.coords))
print("data vars:", list(ds.data_vars))

<xarray.Dataset> Size: 237MB
Dimensions:                           (lat: 4448, lon: 4448)
Coordinates:
  * lat                               (lat) float32 18kB -15.0 -15.0 ... -30.0
  * lon                               (lon) float32 18kB -60.0 -60.0 ... -45.0
Data variables:
    WaterDetection_NEARINTERPOLATION  (lat, lon) float32 79MB ...
    WaterDetection                    (lat, lon) float32 79MB ...
    QualityFlag                       (lat, lon) float32 79MB ...
    quality_information               |S1 1B ...
Attributes: (12/50)
    Conventions:                    CF-1.8, ACDD-1.3
    standard_name_vocabulary:       CF Standard Name Table (version 69, 15 Oc...
    institution:                    DOC/NOAA/NESDIS/OSPO > Office of Satellit...
    naming_authority:               gov.noaa.nesdis.ncei
    history:                        Joint VIIRSABI Flood Detection Version: v1r1
    processing_level:               NOAA Level 4
    ...                             ...
    title:    

## Section 2 — Decode `WaterDetection`

Values 0-99 are categorical (land/vegetation/snow/ice/cloud/...). Values 100-200 are water fraction, where
`percent_water = value - 100` (100 → 0%, 200 → 100%). Mask out the categorical band before rendering.


In [29]:
def load_noaa_water_pct(nc_path):
    ds = xr.open_dataset(nc_path)
    wd = ds["WaterDetection"]
    water_pct = wd.where(wd >= 100) - 100  # 0-100 % water; NaN elsewhere
    return water_pct, ds


def clip_to_aoi(water_pct_da, bbox=AOI_BBOX):
    minx, miny, maxx, maxy = bbox
    lat_name = "lat" if "lat" in water_pct_da.coords else "latitude"
    lon_name = "lon" if "lon" in water_pct_da.coords else "longitude"
    lat_slice = slice(maxy, miny) if water_pct_da[lat_name][0] > water_pct_da[lat_name][-1] else slice(miny, maxy)
    return water_pct_da.sel({lat_name: lat_slice, lon_name: slice(minx, maxx)})


## Section 3 — Render frames (date label)

One shared color ramp (0-100% water) so every frame you add later looks consistent.


In [30]:
CMAP = mcolors.LinearSegmentedColormap.from_list(
    "water_fraction", ["#f5f0e6", "#8ec6ff", "#0b3d91"]  # dry -> partial -> fully flooded
)
NORM = mcolors.Normalize(vmin=0, vmax=100)


def render_frame(water_pct_da, date_str, out_path):
    fig, ax = plt.subplots(figsize=(6, 6))
    water_pct_da.plot(ax=ax, cmap=CMAP, norm=NORM, add_colorbar=False)
    ax.set_axis_off()
    ax.set_title("")
    fig.savefig(out_path, dpi=150, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

    # Overlay date label with PIL
    img = Image.open(out_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 22)
    except OSError:
        font = ImageFont.load_default()
    draw.rectangle([(10, 10), (230, 45)], fill=(0, 0, 0, 160))
    draw.text((16, 14), date_str, fill="white", font=font)
    img.save(out_path)


frame_paths = []
for date_str, nc_path in noaa_local_paths:
    water_pct, _ = load_noaa_water_pct(nc_path)
    water_pct = clip_to_aoi(water_pct)
    out_path = os.path.join(FRAMES_DIR, f"{date_str}_noaa.png")
    render_frame(water_pct, date_str, out_path)
    frame_paths.append((date_str, out_path))

frame_paths.sort(key=lambda t: t[0])
print(f"{len(frame_paths)} frame(s) rendered")


1 frame(s) rendered


## Section 4 — Assemble to GIF / MP4

With only one frame this just makes a static image; add more dates to `NOAA_MANIFEST` above to get real motion.


In [31]:
ordered_pngs = [p for _, p in frame_paths]
gif_path = os.path.join(OUT_DIR, "viirs_flood_timelapse_noaa.gif")
mp4_path = os.path.join(OUT_DIR, "viirs_flood_timelapse_noaa.mp4")

try:
    import leafmap
    leafmap.images_to_gif(ordered_pngs, gif_path, fps=2, mp4=True)
except Exception as e:
    print("leafmap path failed, falling back to imageio:", e)
    import imageio.v2 as imageio
    frames = [imageio.imread(p) for p in ordered_pngs]
    imageio.mimsave(gif_path, frames, fps=2)
    try:
        imageio.mimsave(mp4_path, frames, fps=2)
    except Exception as e2:
        print("MP4 export skipped (install imageio-ffmpeg):", e2)

print("Saved:", gif_path)
print("Saved:", mp4_path if os.path.exists(mp4_path) else "(mp4 not generated, see log above)")


leafmap path failed, falling back to imageio: No module named 'leafmap'
Saved: output/viirs_flood_timelapse_noaa.gif
Saved: output/viirs_flood_timelapse_noaa.mp4


## Next steps

- Add more dates to `NOAA_MANIFEST` once this one frame renders correctly — that's the whole unlock for real
  motion in the GIF.
- When you're ready to fold this into the org project, the post-gap (PC Pro) half slots in as its own
  fetch + render step feeding the same `frame_paths` list — nothing here needs to change to support that later.
